# LUNA16 Dataset Structure Explorer
Run this on Kaggle with the LUNA16 dataset attached.
Copy the full output and share it back.

In [1]:
import os
import glob
import pandas as pd

# ── 1. Find all dataset roots under /kaggle/input ──
INPUT = '/kaggle/input'
print('=== /kaggle/input top-level ===')
for item in sorted(os.listdir(INPUT)):
    full = os.path.join(INPUT, item)
    if os.path.isdir(full):
        n = sum(1 for _ in os.scandir(full))
        print(f'  [DIR]  {item}/  ({n} children)')
    else:
        sz = os.path.getsize(full)
        print(f'  [FILE] {item}  ({sz/1e6:.1f} MB)')

# ── 2. Recurse 3 levels deep ──
print('\n=== Directory tree (depth=3) ===')
for root, dirs, files in os.walk(INPUT):
    depth = root.replace(INPUT, '').count(os.sep)
    if depth > 3:
        dirs.clear()
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth <= 2:
        sub_indent = '  ' * (depth + 1)
        # show first 20 files, then count
        for f in sorted(files)[:20]:
            sz = os.path.getsize(os.path.join(root, f))
            print(f'{sub_indent}{f}  ({sz/1e6:.1f} MB)')
        if len(files) > 20:
            print(f'{sub_indent}... and {len(files)-20} more files')
        if not files and not dirs:
            print(f'{sub_indent}(empty)')
    else:
        sub_indent = '  ' * (depth + 1)
        print(f'{sub_indent}({len(files)} files, {len(dirs)} dirs)')

=== /kaggle/input top-level ===
  [DIR]  datasets/  (1 children)

=== Directory tree (depth=3) ===
input/
  datasets/
    avc0706/
      luna16/
        (3 files, 8 dirs)


In [2]:
# ── 3. Find and preview all CSV files ──
print('=== CSV files found ===')
csvs = glob.glob(os.path.join(INPUT, '**', '*.csv'), recursive=True)
for csv_path in sorted(csvs):
    rel = os.path.relpath(csv_path, INPUT)
    sz = os.path.getsize(csv_path) / 1e6
    print(f'\n--- {rel} ({sz:.2f} MB) ---')
    df = pd.read_csv(csv_path, nrows=5)
    print(f'Columns: {list(df.columns)}')
    print(f'Shape (first 5 rows preview): {df.shape}')
    print(df.to_string(index=False))
    # full row count
    total = sum(1 for _ in open(csv_path)) - 1
    print(f'Total rows: {total}')

=== CSV files found ===

--- datasets/avc0706/luna16/annotations.csv (0.14 MB) ---
Columns: ['seriesuid', 'coordX', 'coordY', 'coordZ', 'diameter_mm']
Shape (first 5 rows preview): (5, 5)
                                                       seriesuid      coordX      coordY      coordZ  diameter_mm
1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222365663678666836860 -128.699421 -175.319272 -298.387506     5.651471
1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222365663678666836860  103.783651 -211.925149 -227.121250     4.224708
1.3.6.1.4.1.14519.5.2.1.6279.6001.100398138793540579077826395208   69.639017 -140.944586  876.374496     5.786348
1.3.6.1.4.1.14519.5.2.1.6279.6001.100621383016233746780170740405  -24.013824  192.102405 -391.081276     8.143262
1.3.6.1.4.1.14519.5.2.1.6279.6001.100621383016233746780170740405    2.441547  172.464881 -405.493732    18.545150
Total rows: 1186

--- datasets/avc0706/luna16/candidates.csv (55.43 MB) ---
Columns: ['seriesuid', 'coordX', 'coordY', 'coordZ',

In [3]:
# ── 4. Find .mhd files and count per subset ──
print('=== .mhd files per directory ===')
mhd_files = glob.glob(os.path.join(INPUT, '**', '*.mhd'), recursive=True)
print(f'Total .mhd files found: {len(mhd_files)}')

from collections import Counter
parent_counts = Counter()
for f in mhd_files:
    parent = os.path.basename(os.path.dirname(f))
    parent_counts[parent] += 1

for parent, count in sorted(parent_counts.items()):
    print(f'  {parent}: {count} scans')

# Show one sample .mhd header
if mhd_files:
    sample = sorted(mhd_files)[0]
    print(f'\n=== Sample .mhd header: {os.path.basename(sample)} ===')
    with open(sample, 'r') as f:
        print(f.read())

=== .mhd files per directory ===
Total .mhd files found: 1333
  seg-lungs-LUNA16: 888 scans
  subset0: 89 scans
  subset1: 89 scans
  subset2: 89 scans
  subset3: 89 scans
  subset4: 89 scans

=== Sample .mhd header: 1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222365663678666836860.mhd ===
ObjectType = Image
NDims = 3
BinaryData = True
BinaryDataByteOrderMSB = False
CompressedData = True
CompressedDataSize = 451779
TransformMatrix = 1 0 0 0 1 0 0 0 1
Offset = -157.67773 -311.67773 -438.39999999999998
CenterOfRotation = 0 0 0
AnatomicalOrientation = RAI
ElementSpacing = 0.64453125 0.64453125 1.7999999523162842
DimSize = 512 512 194
ElementType = MET_SHORT
ElementDataFile = 1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222365663678666836860.zraw



In [4]:
# ── 5. Check for lung segmentation files ──
print('=== Lung segmentation files ===')
seg_dirs = glob.glob(os.path.join(INPUT, '**', '*seg*'), recursive=True)
for s in sorted(seg_dirs)[:10]:
    rel = os.path.relpath(s, INPUT)
    if os.path.isdir(s):
        n = len(os.listdir(s))
        print(f'  [DIR] {rel}/ ({n} children)')
    else:
        print(f'  [FILE] {rel} ({os.path.getsize(s)/1e6:.1f} MB)')

# Also check for any zip files
print('\n=== Zip files ===')
zips = glob.glob(os.path.join(INPUT, '**', '*.zip'), recursive=True)
for z in sorted(zips):
    rel = os.path.relpath(z, INPUT)
    print(f'  {rel} ({os.path.getsize(z)/1e6:.1f} MB)')
if not zips:
    print('  (none found)')

=== Lung segmentation files ===
  [DIR] datasets/avc0706/luna16/seg-lungs-LUNA16/ (1 children)
  [DIR] datasets/avc0706/luna16/seg-lungs-LUNA16/seg-lungs-LUNA16/ (1776 children)

=== Zip files ===
  (none found)
